# DistilBERT on WikiText — encoder activations in memory

Every transformer block outputs `(batch, seq_len, hidden)`, so a text run costs
far more per sample than a vision one. Downloads on first run: WikiText-2
(~5 MB) and DistilBERT weights (~250 MB).

In [ ]:
import torch
from datasets import load_dataset
from torch.utils.data import Dataset
from transformers import AutoModel, AutoTokenizer

from nnact import ActivationMapper, Sample

MODEL = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)

In [ ]:
class WikiTextSamples(Dataset[Sample]):
    """WikiText passages, tokenised to a fixed length.

    nnact stacks Sample.data across a batch, so every row must be the same
    shape: pad to a fixed max_length rather than per batch. Attention masks
    are kept on the instance for pooling later.
    """

    def __init__(self, tokenizer, n: int = 256, max_length: int = 64) -> None:
        raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
        # Drop blanks and the "= Heading =" lines the raw dump is full of.
        self.texts = [
            t.strip()
            for t in raw["text"]
            if len(t.strip()) > 120 and not t.strip().startswith("=")
        ][:n]
        self.encoded = tokenizer(
            self.texts,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Sample:
        return Sample(id=f"wiki_{idx:04d}", data=self.encoded["input_ids"][idx])


dataset = WikiTextSamples(tokenizer, n=256, max_length=64)
print(f"{len(dataset)} passages | input_ids {tuple(dataset[0].data.shape)}")
print(f"real tokens in first: {int(dataset.encoded['attention_mask'][0].sum())}")
print(dataset.texts[0][:90], "...")

In [ ]:
mapper = ActivationMapper(model)

# depth=3 reaches the individual blocks; depth=1 would only show
# "embeddings" and "transformer".
mapper.summary(depth=3).head(12)

In [ ]:
# Embeddings, an early block, a middle block, and the last one.
store = mapper.map(
    dataset,
    ["embeddings", "transformer.layer.0", "transformer.layer.3", "transformer.layer.5"],
    batch_size=32,
)
store.metadata

In [ ]:
# (n_samples, seq_len, hidden) per layer - seq_len is why text is expensive.
store.summary()

In [ ]:
# Row i of a layer is one passage's whole token sequence.
last = store.activations["transformer.layer.5"]
print("layer 5 stacked:", tuple(last.shape))

# Most uses want one vector per passage. Mean-pool over real tokens only,
# so padding does not drag the average toward zero.
mask = dataset.encoded["attention_mask"].unsqueeze(-1).float()
pooled = (last * mask).sum(1) / mask.sum(1)
print("mean-pooled:", tuple(pooled.shape), "| CLS:", tuple(last[:, 0].shape))

In [ ]:
# How far each layer's representation sits from the final one.
for name in store.layer_names:
    vec = (store.activations[name] * mask).sum(1) / mask.sum(1)
    sim = torch.nn.functional.cosine_similarity(vec, pooled, dim=1).mean()
    print(f"  {name:<22} cos(layer, final) = {sim:.3f}")